# 05 - Feature Engineering

## Goal

The goal of this notebook is to engineer meaningful possession-level features from the event data.

These features will describe the characteristics of each attacking possession and will be used as predictors in the machine learning models.

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
import os

In [3]:
events_df = pd.read_csv("../data/raw/fawsl_events.csv")

possession_df = pd.read_csv("../data/processed/possession_dataset.csv")

C:\Users\Korisnik\AppData\Local\Temp\ipykernel_13952\3857023784.py:1: DtypeWarning: Columns (0: bad_behaviour_card, 1: ball_recovery_offensive, 2: block_offensive, 3: dribble_nutmeg, 4: dribble_overrun, 5: foul_committed_advantage, 6: foul_committed_card, 7: foul_committed_type, 8: foul_won_advantage, 9: goalkeeper_shot_saved_to_post, 10: injury_stoppage_in_chain, 11: pass_deflected, 12: pass_straight, 13: shot_one_on_one, 14: shot_open_goal, 15: shot_saved_to_post, 16: block_deflection, 17: miscontrol_aerial_won, 18: pass_outswinging, 19: shot_deflected, 20: goalkeeper_shot_saved_off_target, 21: pass_miscommunication, 22: pass_no_touch, 23: shot_redirect, 24: shot_saved_off_target, 25: clearance_other, 26: foul_committed_offensive, 27: dribble_no_touch, 28: block_save_block, 29: foul_committed_penalty, 30: foul_won_penalty, 31: player_off_permanent, 32: goalkeeper_lost_out, 33: goalkeeper_punched_out, 34: goalkeeper_success_out, 35: goalkeeper_success_in_play, 36: goalkeeper_penalty_s

In [4]:
events_df.shape

(495189, 115)

In [5]:
possession_df.shape

(24814, 7)

## Feature 1: Possession Duration

In [6]:
possession_duration = (
    events_df
    .groupby(["match_id", "possession"])["duration"]
    .sum()
    .reset_index(name="duration")
)

In [7]:
possession_duration.head()

,match_id,possession,duration
0,3912592,1,0.000000
1,3912592,2,21.920786
2,3912592,3,49.032388
3,3912592,4,2.093451
4,3912592,5,25.660795


In [8]:
possession_df = possession_df.merge(

    possession_duration,

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [9]:
possession_df.head()

,match_id,possession,num_events,team,possession_team,type,ends_with_shot,duration
0,3912592,1,4,Manchester United W,Aston Villa W,Half Start,0,0.000000
1,3912592,2,21,Manchester United W,Manchester United W,Ball Receipt*,0,21.920786
2,3912592,3,68,Manchester United W,Aston Villa W,Pressure,0,49.032388
3,3912592,4,5,Manchester United W,Manchester United W,Foul Won,0,2.093451
4,3912592,5,28,Manchester United W,Manchester United W,Ball Receipt*,0,25.660795


## Feature 2: Number of Passes

In [10]:
passes = (
    events_df[
        events_df["type"] == "Pass"
    ]
    .groupby(
        ["match_id","possession"]
    )
    .size()
    .reset_index(name="num_passes")
)

In [11]:
possession_df = possession_df.merge(

    passes,

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [12]:
possession_df["num_passes"] = (
    possession_df["num_passes"]
    .fillna(0)
)

## Feature 3: Number of Carries

In [13]:
carries = (
    events_df[
        events_df["type"] == "Carry"
    ]
    .groupby(
        ["match_id","possession"]
    )
    .size()
    .reset_index(name="num_carries")
)

In [14]:
possession_df = possession_df.merge(

    carries,

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [15]:
possession_df["num_carries"] = (
    possession_df["num_carries"]
    .fillna(0)
)

## Feature 4: Number of Dribbles

In [16]:
dribbles = (
    events_df[
        events_df["type"] == "Dribble"
    ]
    .groupby(
        ["match_id","possession"]
    )
    .size()
    .reset_index(name="num_dribbles")
)

In [17]:
possession_df = possession_df.merge(

    dribbles,

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [18]:
possession_df["num_dribbles"] = (
    possession_df["num_dribbles"]
    .fillna(0)
)

## Feature 5: Number of Duels

In [19]:
duels = (
    events_df[
        events_df["type"] == "Duel"
    ]
    .groupby(
        ["match_id", "possession"]
    )
    .size()
    .reset_index(name="num_duels")
)

possession_df = possession_df.merge(
    duels,
    on=["match_id", "possession"],
    how="left"
)

possession_df["num_duels"] = (
    possession_df["num_duels"]
    .fillna(0)
)

## Feature 6: Number of Players Involved

In [20]:
players = (
    events_df
    .groupby(
        ["match_id", "possession"]
    )["player"]
    .nunique()
    .reset_index(name="num_players")
)

possession_df = possession_df.merge(
    players,
    on=["match_id", "possession"],
    how="left"
)

## Feature 7: Pressure Events

In [21]:
pressure = (
    events_df[
        events_df["under_pressure"] == True
    ]
    .groupby(
        ["match_id", "possession"]
    )
    .size()
    .reset_index(name="pressure_events")
)

possession_df = possession_df.merge(
    pressure,
    on=["match_id", "possession"],
    how="left"
)

possession_df["pressure_events"] = (
    possession_df["pressure_events"]
    .fillna(0)
)

## Feature 8: Miscontrols

In [22]:
miscontrols = (
    events_df[
        events_df["type"] == "Miscontrol"
    ]
    .groupby(
        ["match_id", "possession"]
    )
    .size()
    .reset_index(name="num_miscontrols")
)

possession_df = possession_df.merge(
    miscontrols,
    on=["match_id", "possession"],
    how="left"
)

possession_df["num_miscontrols"] = (
    possession_df["num_miscontrols"]
    .fillna(0)
)

## Feature 9: Dispossessions

In [23]:
dispossessed = (
    events_df[
        events_df["type"] == "Dispossessed"
    ]
    .groupby(
        ["match_id", "possession"]
    )
    .size()
    .reset_index(name="num_dispossessed")
)

possession_df = possession_df.merge(
    dispossessed,
    on=["match_id", "possession"],
    how="left"
)

possession_df["num_dispossessed"] = (
    possession_df["num_dispossessed"]
    .fillna(0)
)

## Feature: Possession Starting Location

In [24]:
events_df["location"].head()

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: location, dtype: str

In [25]:
events_df["location"] = events_df["location"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [26]:
events_df["location"].head(10)

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6    [60.0, 40.0]
7    [54.5, 44.4]
8    [34.2, 53.3]
9    [12.2, 46.7]
Name: location, dtype: object

In [27]:
start_location = (
    events_df
    .sort_values(
        [
            "match_id",
            "possession",
            "minute",
            "second"
        ]
    )
    .groupby(
        [
            "match_id",
            "possession"
        ]
    )
    .first()
    .reset_index()
)

In [28]:
start_location["start_x"] = (
    start_location["location"]
    .apply(
        lambda x: x[0] if isinstance(x, list) else np.nan
    )
)

start_location["start_y"] = (
    start_location["location"]
    .apply(
        lambda x: x[1] if isinstance(x, list) else np.nan
    )
)

In [29]:
start_location[
    [
        "match_id",
        "possession",
        "start_x",
        "start_y"
    ]
].head()

,match_id,possession,start_x,start_y
0,3912592,1,NaN,NaN
1,3912592,2,61.0,40.1
2,3912592,3,48.1,0.1
3,3912592,4,69.0,78.8
4,3912592,5,67.6,76.0


In [30]:
start_location["start_x"].isna().mean()

np.float64(0.005319577657773837)

In [31]:
start_location["start_y"].isna().mean()

np.float64(0.005319577657773837)

In [32]:
possession_df = possession_df.merge(

    start_location[
        [
            "match_id",
            "possession",
            "start_x",
            "start_y"
        ]
    ],

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [33]:
end_location = (
    events_df
    .sort_values(
        [
            "match_id",
            "possession",
            "minute",
            "second"
        ]
    )
    .groupby(
        [
            "match_id",
            "possession"
        ]
    )
    .last()
    .reset_index()
)

In [34]:
end_location["end_x"] = (
    end_location["location"]
    .apply(
        lambda x: x[0] if isinstance(x, list) else np.nan
    )
)

end_location["end_y"] = (
    end_location["location"]
    .apply(
        lambda x: x[1] if isinstance(x, list) else np.nan
    )
)

In [35]:
end_location[
    [
        "match_id",
        "possession",
        "end_x",
        "end_y"
    ]
].head()

,match_id,possession,end_x,end_y
0,3912592,1,NaN,NaN
1,3912592,2,64.4,79.6
2,3912592,3,65.3,69.3
3,3912592,4,67.4,76.0
4,3912592,5,112.6,42.2


In [36]:
possession_df.columns

Index(['match_id', 'possession', 'num_events', 'team', 'possession_team',
       'type', 'ends_with_shot', 'duration', 'num_passes', 'num_carries',
       'num_dribbles', 'num_duels', 'num_players', 'pressure_events',
       'num_miscontrols', 'num_dispossessed', 'start_x', 'start_y'],
      dtype='str')

In [37]:
possession_df = possession_df.merge(

    end_location[
        [
            "match_id",
            "possession",
            "end_x",
            "end_y"
        ]
    ],

    on=[
        "match_id",
        "possession"
    ],

    how="left"
)

In [38]:
possession_df["progression"] = (
    possession_df["end_x"]
    -
    possession_df["start_x"]
)

In [39]:
possession_df[
    [
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "progression"
    ]
].head(10)

,start_x,start_y,end_x,end_y,progression
0,NaN,NaN,NaN,NaN,NaN
1,61.0,40.1,64.4,79.6,3.4
2,48.1,0.1,65.3,69.3,17.2
3,69.0,78.8,67.4,76.0,-1.6
4,67.6,76.0,112.6,42.2,45.0
5,1.4,40.2,117.8,67.0,116.4
6,120.0,80.0,118.9,18.5,-1.1
7,7.0,36.1,82.3,1.9,75.3
8,37.3,80.0,85.5,2.9,48.2
9,37.1,80.0,97.7,60.2,60.6


In [40]:
possession_df.shape

(24814, 21)

In [41]:
possession_df.isnull().sum().sort_values(ascending=False).head(15)

start_x            132
progression        132
end_y              132
end_x              132
start_y            132
possession_team      0
team                 0
num_events           0
possession           0
match_id             0
type                 0
duration             0
ends_with_shot       0
num_players          0
num_duels            0
dtype: int64

In [42]:
possession_df = possession_df.drop(
    columns=["type"]
)

In [43]:
possession_df.duplicated().sum()

np.int64(0)

In [44]:
possession_df["ends_with_shot"].value_counts()

ends_with_shot
0    24381
1      433
Name: count, dtype: int64

In [45]:
possession_df["ends_with_shot"].value_counts(normalize=True) * 100

ends_with_shot
0    98.255017
1     1.744983
Name: proportion, dtype: float64

In [46]:
possession_df["num_turnovers"] = (
    possession_df["num_miscontrols"]
    +
    possession_df["num_dispossessed"]
)

In [47]:
possession_df["pass_ratio"] = (
    possession_df["num_passes"]
    /
    possession_df["num_events"]
)

In [48]:
possession_df["progression_rate"] = (
    possession_df["progression"]
    /
    possession_df["duration"]
)

In [49]:
possession_df.shape

(24814, 23)

In [50]:
possession_df.columns

Index(['match_id', 'possession', 'num_events', 'team', 'possession_team',
       'ends_with_shot', 'duration', 'num_passes', 'num_carries',
       'num_dribbles', 'num_duels', 'num_players', 'pressure_events',
       'num_miscontrols', 'num_dispossessed', 'start_x', 'start_y', 'end_x',
       'end_y', 'progression', 'num_turnovers', 'pass_ratio',
       'progression_rate'],
      dtype='str')

## Final Feature Set

The possession-level dataset was enriched with additional attacking and spatial features to support predictive modeling.

The final features include:

- Possession structure:
  - number of events
  - possession duration
  - number of involved players

- Attacking actions:
  - passes
  - carries
  - dribbles
  - duels

- Defensive pressure and possession losses:
  - pressure events
  - miscontrols
  - dispossessions
  - total turnovers

- Spatial progression:
  - starting and ending coordinates
  - ball progression
  - progression rate

- Derived passing behaviour:
  - pass ratio

The target variable `ends_with_shot` indicates whether a possession resulted in a shot attempt.

This dataset will be used for exploratory data analysis and the first predictive model focused on shot creation probability.

In [51]:
possession_df.to_csv(
    "../data/processed/possession_features.csv",
    index=False
)

In [52]:
os.path.getsize(
    "../data/processed/possession_features.csv"
) / (1024*1024)

3.806666374206543